In [3]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt

()

In [1]:

# Define the function for adding depth parameters and flattening attributes
def add_depth_parameters(ds, water_depth_from_mooring_diagram, water_depth_from_ship_uncorrected,
                         water_depth_from_ship_corrected, instrument_height_above_bottom):
    # Define the depth parameters dictionary
    depth_parameters = {
        # consult mooring diagram
        'water_depth_from_mooring_diagram': water_depth_from_mooring_diagram,  
        # uncorrected water depth
        'water_depth_from_ship_uncorrected': water_depth_from_ship_uncorrected,
        # Replace with actual value, best water depth
        'water_depth_from_ship_corrected': water_depth_from_ship_corrected,        
        # instrument_depth_from_mooring_diagram = diagram depth  - height above bottom 
        'instrument_depth_from_mooring_diagram': water_depth_from_mooring_diagram - instrument_height_above_bottom,  
        # instrument_depth_from_mooring_log = corrected depth (4538.97 m) - height above bottom (39 m) 
        'instrument_depth_from_mooring_log': water_depth_from_ship_corrected - instrument_height_above_bottom,
        'instrument_height_above_bottom': 39     
        #  Anchor (1) + chain (5) + nystron (20) + chain (5) + releases (1) + chain (5) 
        # + 6 terminations at 0.25 ea (1.5) 
        # + distance from termination on SBE cage to sensor (0.5) = 39 m
    }

    # Update the attributes with the depth parameters
    ds.attrs.update(depth_parameters)

    # Convert dataset to JSON
    ds_json = ds.to_dict()

    # Flatten the JSON object
    def flatten_dict(d, parent_key='', sep='_'):
        items = []
        for k, v in d.items():
            new_key = parent_key + sep + k if parent_key else k
            if isinstance(v, dict):
                items.extend(flatten_dict(v, new_key, sep=sep).items())
            else:
                items.append((new_key, v))
        return dict(items)

    ds_json_flat = flatten_dict(ds_json)

    # Update the attributes with the flattened JSON object
    ds.attrs.update(ds_json_flat)

    # Remove the 'attributes' attribute
    if 'attributes' in ds.attrs:
        del ds.attrs['attributes']

    return ds

In [10]:
netcdf_path = '/Users/yugao/UOP/ORS-processing/data/processed/stratus/'
# Load the dataset
# Replace 'your_data.nc' with the path to your dataset file
ds = xr.open_dataset(netcdf_path + '/stratus12_sbe16_1876.nc')

FileNotFoundError: [Errno 2] No such file or directory: '/Users/yugao/UOP/ORS-processing/data/processed/stratus/stratus12_sbe16_1876.nc'

In [9]:
# Calculate the difference in temperature
temp_diff = ds['temp'].diff('time')

# Find the indices where the absolute temperature difference exceeds a threshold
threshold = 2.0  # Adjust the threshold as needed
deploy_indices = np.where(np.abs(temp_diff) > threshold)[0]

# Filter out false signals (e.g., small temporary drops before the actual deployment)
filtered_indices = []
for index in deploy_indices:
    # Check if the temperature drop lasts for a certain number of time steps (e.g., 24 hours)
    if np.all(np.abs(temp_diff[index:index+24]) > threshold):
        filtered_indices.append(index)

# Take the first deployment index as the actual deployment event
deploy_index = filtered_indices[0]

# Crop the data around the deployment event
start_index = deploy_index - 30 * 24  # 30 days before in hours
end_index = deploy_index + 30 * 24    # 30 days after in hours
cropped_ds = ds.isel(time=slice(start_index, end_index))

# Add depth parameters and flatten attributes
cropped_ds = add_depth_parameters(cropped_ds, water_depth_from_mooring_diagram, water_depth_from_ship_uncorrected,
                                   water_depth_from_ship_corrected, instrument_height_above_bottom)

# Plot temperature and salinity data before and after cropping
fig, axs = plt.subplots(2, 1, figsize=(10, 8))

# Before cropping
ds['temp'].plot(ax=axs[0], label='Before cropping')
ds['sal'].plot(ax=axs[1], label='Before cropping')

# After cropping
cropped_ds['temp'].plot(ax=axs[0], label='After cropping')
cropped_ds['sal'].plot(ax=axs[1], label='After cropping')

# Add legends
axs[0].legend()
axs[1].legend()

# Show the plot
plt.tight_layout()
plt.show()


IndexError: list index out of range